# Hirriririir Thigh Segmentation — Fat / Water input (Lambda)

Corrected version: feeds **Dixon Fat or Dixon Water** images to the SegResNetDS model,
matching the training data the checkpoint was actually trained on.

Previous runs used fat-fraction images (values 0–100), which are outside the model's
expected intensity range ([147, 2696] from `hyper_parameters.yaml`).

## Data expected on Lambda
```
~/myosegmenTUM/{subject}/ImageData/{subject}_FAT/{subject}_FAT_stack*.nii
~/myosegmenTUM/{subject}/ImageData/{subject}_WATER/{subject}_WATER_stack*.nii
```

## Steps
1. Set `MODALITY = 'FAT'` or `'WATER'` in the config cell
2. Run all cells
3. Results saved to `~/multimodal_thigh_segs_fat/` or `~/multimodal_thigh_segs_water/`
4. Download: `rsync -avz ubuntu@<IP>:~/multimodal_thigh_segs_fat/ ./eval_notebooks/multimodal_thigh_segs_fat/`

In [1]:
import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'monai', 'SimpleITK'])
# print('Dependencies ready')

In [2]:
import subprocess, os

# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# result = subprocess.run(
#     ['nvidia-smi', '--query-compute-apps=pid,used_memory,process_name',
#      '--format=csv,noheader'],
#     capture_output=True, text=True)
# print('Processes using GPU:')
# print(result.stdout or '  (none)')

# mem = subprocess.run(
#     ['nvidia-smi', '--query-gpu=memory.free,memory.total',
#      '--format=csv,noheader,nounits'],
#     capture_output=True, text=True)
# free, total = [int(x) for x in mem.stdout.strip().split(', ')]
# print(f'GPU memory: {free} MiB free / {total} MiB total')

# if free < 8000:
#     print('\nWARNING: less than 8 GB free.')
#     print('In JupyterLab: Kernel menu → Shut Down All Kernels, then re-open this notebook.')
#     print('Or kill the offending PID above:  import os; os.system("sudo kill <PID>")')

In [10]:
!pwd

/c/Projects/dissector/eval_notebooks


In [12]:
import glob, os
import numpy as np
#import torch
import SimpleITK as sitk
#from monai.networks.nets import SegResNetDS
#from monai.inferers import sliding_window_inference

# ── Config ───────────────────────────────────────────────────────────────────
MODALITY = 'WATER'   # 'FAT' or 'WATER'

DATA_ROOT   = os.path.expanduser('myosegmenTUM')
CHECKPOINT  = os.path.expanduser('~/pretrained_segmentation_muscle.pt')
OUTPUT_DIR  = os.path.expanduser(f'~/multimodal_thigh_segs_{MODALITY.lower()}')

# Glob: ~/myosegmenTUM/{subject}/ImageData/{subject}_FAT/{subject}_FAT_stack*.nii
IMAGE_GLOB  = os.path.join(DATA_ROOT, '*', 'ImageData',
                           f'*_{MODALITY}', f'*_{MODALITY}_stack*.nii')

# From hyper_parameters.yaml in the Hirriririir repo
TARGET_SPACING    = (0.7813, 0.7813, 4.0)
ROI_SIZE          = [336, 336, 88]
INTENSITY_LOWER   = 146.94    # intensity_bounds[0]
INTENSITY_UPPER   = 2695.88   # intensity_bounds[1]

DEVICE = 'bla'#'cuda' if torch.cuda.is_available() else 'cpu'

LABEL_MAP = {
    1:  'Sartorius',
    2:  'Rectus_Femoris',
    3:  'Vastus_Lateralis',
    4:  'Vastus_Intermedius',
    5:  'Vastus_Medialis',
    6:  'Adductor_Magnus',
    7:  'Gracilis',
    8:  'Biceps_Femoris_Long',
    9:  'Semitendinosus',
    10: 'Semimembranosus',
    11: 'Biceps_Femoris_Short',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Modality:          ', MODALITY)
print('Device:            ', DEVICE)
print('Checkpoint exists: ', os.path.exists(CHECKPOINT))
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} {MODALITY} stacks')
if image_files:
    print('  First:', image_files[0])
    print(image_files)

Modality:           WATER
Device:             bla
Checkpoint exists:  True
Found 46 WATER stacks
  First: myosegmenTUM\HV001_1\ImageData\HV001_1_WATER\HV001_1_WATER_stack1.nii
['myosegmenTUM\\HV001_1\\ImageData\\HV001_1_WATER\\HV001_1_WATER_stack1.nii', 'myosegmenTUM\\HV001_1\\ImageData\\HV001_1_WATER\\HV001_1_WATER_stack2.nii', 'myosegmenTUM\\HV001_2\\ImageData\\HV001_2_WATER\\HV001_2_WATER_stack1.nii', 'myosegmenTUM\\HV001_2\\ImageData\\HV001_2_WATER\\HV001_2_WATER_stack2.nii', 'myosegmenTUM\\HV001_3\\ImageData\\HV001_3_WATER\\HV001_3_WATER_stack1.nii', 'myosegmenTUM\\HV001_3\\ImageData\\HV001_3_WATER\\HV001_3_WATER_stack2.nii', 'myosegmenTUM\\HV002_1\\ImageData\\HV002_1_WATER\\HV002_1_WATER_stack1.nii', 'myosegmenTUM\\HV002_1\\ImageData\\HV002_1_WATER\\HV002_1_WATER_stack2.nii', 'myosegmenTUM\\HV002_2\\ImageData\\HV002_2_WATER\\HV002_2_WATER_stack1.nii', 'myosegmenTUM\\HV002_2\\ImageData\\HV002_2_WATER\\HV002_2_WATER_stack2.nii', 'myosegmenTUM\\HV002_3\\ImageData\\HV002_3_WATER\\HV0

In [7]:
# Download checkpoint if not already present (~345 MB)
import urllib.request

if not os.path.exists(CHECKPOINT):
    print('Downloading checkpoint...')
    url = ('https://github.com/Hirriririir/Multimodal-Multiethnic-Thigh-Muscle-MRI-analysis'
           '/releases/download/1.0/pretrained_segmentation_muscle.pt')
    urllib.request.urlretrieve(url, CHECKPOINT)
    print(f'Done ({os.path.getsize(CHECKPOINT) // 1_000_000} MB)')
else:
    print('Checkpoint already present')

KeyboardInterrupt: 

In [8]:
model = SegResNetDS(
    spatial_dims=3,
    in_channels=1,
    out_channels=12,
    init_filters=32,
    blocks_down=(1, 2, 2, 4, 4),
    dsdepth=4,
    norm='INSTANCE',
    resolution=TARGET_SPACING,
)
ckpt  = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
state = (ckpt.get('state_dict') or ckpt.get('network_weights') or ckpt
         if isinstance(ckpt, dict) else ckpt)
missing, unexpected = model.load_state_dict(state, strict=False)
if missing:    print('Missing:   ', missing[:5])
if unexpected: print('Unexpected:', unexpected[:5])
model = model.to(DEVICE).eval()
print('Model ready on', DEVICE)

NameError: name 'SegResNetDS' is not defined

In [ ]:
def resample_sitk(sitk_img, new_spacing, interpolator=sitk.sitkLinear):
    orig_spacing = sitk_img.GetSpacing()
    orig_size    = sitk_img.GetSize()
    new_size = [
        int(round(orig_size[i] * orig_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]
    r = sitk.ResampleImageFilter()
    r.SetOutputSpacing(new_spacing)
    r.SetSize(new_size)
    r.SetOutputDirection(sitk_img.GetDirection())
    r.SetOutputOrigin(sitk_img.GetOrigin())
    r.SetTransform(sitk.Transform())
    r.SetDefaultPixelValue(0)
    r.SetInterpolator(interpolator)
    return r.Execute(sitk_img)


def preprocess(nii_path):
    img = sitk.ReadImage(nii_path, sitk.sitkFloat32)
    res = resample_sitk(img, TARGET_SPACING)

    # SimpleITK returns (nz, ny, nx); MONAI/nibabel convention is (nx, ny, nz).
    # The model was trained with MONAI, so transpose to match.
    arr = sitk.GetArrayFromImage(res).astype(np.float32).transpose(2, 1, 0)

    arr  = np.clip(arr, INTENSITY_LOWER, INTENSITY_UPPER)
    mean = arr.mean()
    std  = arr.std()
    arr  = (arr - mean) / (std + 1e-8)

    return arr, res, img


def infer_volume(arr):
    t = torch.tensor(arr[None, None]).float().to(DEVICE)
    with torch.no_grad():
        out = sliding_window_inference(
            t, roi_size=ROI_SIZE, sw_batch_size=1,
            predictor=model, overlap=0.5, mode='gaussian'
        )
    logits = out[0] if isinstance(out, (list, tuple)) else out
    return torch.argmax(logits, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)


print('Preprocessing and inference functions defined')

In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_thigh_seg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    arr, res_ref, orig = preprocess(nii_path)
    print(f'  Array shape (nx, ny, nz): {arr.shape}')
    print(f'  Intensity after clip+norm: mean={arr.mean():.3f}  std={arr.std():.3f}')

    pred = infer_volume(arr)
    print(f'  Labels present: {sorted(np.unique(pred).tolist())}')

    # Transpose back from MONAI (nx, ny, nz) to SimpleITK (nz, ny, nx)
    pred_sitk = sitk.GetImageFromArray(pred.transpose(2, 1, 0))
    pred_sitk.CopyInformation(res_ref)
    pred_orig = sitk.Resample(pred_sitk, orig,
                               sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    sitk.WriteImage(pred_orig, out_path)
    print(f'  Saved -> {out_path}')

    pred_arr = sitk.GetArrayFromImage(pred_orig)
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    print(f'  {"-"*45}')
    for idx, name in LABEL_MAP.items():
        n = int((pred_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')